# Sprint 2 Role 2 — Data Validation

Re-runnable companion to `docs/data_validation.md` (issue #21). Loads the
merged `opportunity_df` from `src.opportunity_cleaner.build_opportunity_df()`,
applies Yixiao's `clean_opportunity_df` to get the Sprint 2
`cleaned_opportunity_df`, and calls each function in `src.data_validator` so
every number in the notes document can be reproduced from a clean kernel.

Run order: setup → missingness → revenue → date/duration → categorical →
data-quality flags → status_reason taxonomy → owner identity → owner
aggregates → field reliability for scoring.

## Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

from src.opportunity_cleaner import build_opportunity_df, clean_opportunity_df
from src.data_validator import (
    Fields,
    apply_revenue_hierarchy,
    build_field_reliability_report,
    build_owner_aggregates,
    classify_opportunity_outcome,
    summarize_missingness,
    validate_categorical_fields,
    validate_date_duration_fields,
    validate_owner_identity,
    validate_quality_flags,
    validate_revenue_fields,
)

opportunity_df, schema_comparison, merge_summary, audit_tables = build_opportunity_df()

# Sprint 2 input: the Week 1 merge plus Yixiao's additive cleaning pass
# (merge-provenance flags, data-quality flags, precomputed authoritative_revenue).
# Every validator below runs on this cleaned frame.
cleaned_opportunity_df = clean_opportunity_df(opportunity_df)
print(
    f"opportunity_df:         {len(opportunity_df):,} rows, "
    f"{opportunity_df.shape[1]} columns"
)
print(
    f"cleaned_opportunity_df: {len(cleaned_opportunity_df):,} rows, "
    f"{cleaned_opportunity_df.shape[1]} columns"
)

opportunity_df:         8,746 rows, 45 columns
cleaned_opportunity_df: 8,746 rows, 58 columns


## 1. Missingness summary

One row per validated business field. The columns most central to scoring
(`status`, `sales_stage`, `opportunity_owner`, `opportunity_manager`) are
100% populated. The 12.2% null on `opportunity_estimated_revenue_base_cad`
is the unmatched opps2 rows, not a defect (see §2).

In [2]:
missingness = summarize_missingness(cleaned_opportunity_df)
missingness

,field,dtype,n_total,n_null,pct_null,n_blank_string,n_zero,n_negative,family
0,total_estimated_revenue,float64,8746,51,0.58,0,291,0,revenue
1,opportunity_estimated_revenue_base_cad,float64,8746,1067,12.20,0,82,0,revenue
2,service_solution_estimated_revenue,float64,8746,3833,43.83,0,148,0,revenue
3,created_on,datetime64[ns],8746,51,0.58,0,0,51,date
4,close_date,datetime64[ns],8746,365,4.17,0,0,365,date
5,revenue_start_date,datetime64[ns],8746,1092,12.49,0,0,1092,date
6,project_duration_number_of_months,float64,8746,852,9.74,0,43,0,duration
7,status,object,8746,0,0.00,0,0,0,categorical
8,status_reason,object,8746,51,0.58,0,0,0,categorical
9,sales_stage,object,8746,0,0.00,0,0,0,categorical


## 2. Revenue field hierarchy

`total_estimated_revenue` is the canonical revenue field; the fallback to
`opportunity_estimated_revenue_base_cad` covers the 51 opps1-exclusive rows
where the primary is null. As of Yixiao's Sprint 2 merge the fallback field
is also carried on matched rows, so the two now overlap on ~7,628 rows and
agree within 5% on ~99.5% of them — a cross-validation that was not possible
in Sprint 1. Service-solution revenue must **not** be summed onto either.

In [3]:
validate_revenue_fields(cleaned_opportunity_df)

F:\capstone\cgi-capstone\src\data_validator.py:713: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  within_5pct = (ratio.fillna(0) <= 0.05).sum()


,metric,value
0,n_total,8746.0
1,n_primary_null,51.0
2,n_fallback_null,1067.0
3,n_service_null,3833.0
4,n_primary_zero,291.0
5,n_fallback_zero,82.0
6,n_primary_negative,0.0
7,n_fallback_negative,0.0
8,n_unscoreable_both_null,0.0
9,n_both_present,7628.0


In [4]:
# Apply the hierarchy and inspect what gets routed to fallback.
# cleaned_opportunity_df already carries a precomputed authoritative_revenue;
# apply_revenue_hierarchy recomputes it from source so the validator stays
# independent of the cleaner's version.
resolved = apply_revenue_hierarchy(cleaned_opportunity_df)
print("revenue_source value counts:")
print(resolved["revenue_source"].value_counts(dropna=False))
print("\nMatches the cleaner's precomputed column on every row:",
      bool((resolved["authoritative_revenue"].fillna(-1)
            == cleaned_opportunity_df["authoritative_revenue"].fillna(-1)).all()))
print("\nFallback rows (showing 5):")
(
    cleaned_opportunity_df.assign(**resolved)
    .loc[resolved["revenue_source"] == "fallback",
         [Fields.OWNER, Fields.STATUS, "authoritative_revenue", "revenue_source"]]
    .head()
)

revenue_source value counts:
revenue_source
primary     8695
fallback      51
Name: count, dtype: int64

Matches the cleaner's precomputed column on every row: True

Fallback rows (showing 5):


,opportunity_owner,status,authoritative_revenue,revenue_source
8695,Indira Foxworth,Won,0.0,fallback
8696,Jordan Jameson,Open,2500000.0,fallback
8697,Jordan Jameson,Open,40792.0,fallback
8698,Jordan Jameson,Open,1249824.0,fallback
8699,Jordan Jameson,Open,500000.0,fallback


## 3. Date and duration validation

Critical findings:
1. Delivery window is 100% resolvable on the Won subset (5,275 via
   `revenue_start_date`, 15 via the `close_date` fallback).
2. `n_close_before_created` is **379** on a calendar-date basis — down from
   the Sprint 1 figure of 3,049, which was a raw-timestamp artefact.
3. Duration outliers > 60 months persist (152 rows); the `clip` in
   `capacity_engine` does not address them.

In [5]:
validate_date_duration_fields(cleaned_opportunity_df)

,metric,value
0,n_total,8746
1,as_of,2026-05-14
2,n_created_on_null,51
3,n_close_date_null,365
4,n_revenue_start_date_null,1092
5,n_duration_null,852
6,n_duration_zero,43
7,n_duration_negative,0
8,n_duration_over_60_months,152
9,n_close_before_created,379


In [6]:
# Investigate close_before_created on a calendar-date basis (the corrected
# comparison). created_on carries a timestamp; close_date is midnight, so
# both must be normalized before comparing — the raw-timestamp version
# inflated this count to 3,049.
created = pd.to_datetime(cleaned_opportunity_df[Fields.CREATED_ON], errors="coerce").dt.normalize()
close = pd.to_datetime(cleaned_opportunity_df[Fields.CLOSE_DATE], errors="coerce").dt.normalize()
delta_days = (close - created).dt.days
anomalies = cleaned_opportunity_df.assign(_delta_days=delta_days).loc[delta_days < 0]
print(f"Rows with close_date < created_on (calendar date): {len(anomalies):,}")
raw_delta = (
    pd.to_datetime(cleaned_opportunity_df[Fields.CLOSE_DATE], errors="coerce")
    - pd.to_datetime(cleaned_opportunity_df[Fields.CREATED_ON], errors="coerce")
).dt.days
print(f"For comparison, raw-timestamp count (the Sprint 1 artefact): {(raw_delta < 0).sum():,}")
print("\nBy status:")
print(anomalies[Fields.STATUS].value_counts())
print("\nDelta distribution (days):")
print(delta_days.loc[delta_days < 0].describe().round(1))

Rows with close_date < created_on (calendar date): 379
For comparison, raw-timestamp count (the Sprint 1 artefact): 3,049

By status:
status
Won       377
Closed      2
Name: count, dtype: int64

Delta distribution (days):
count    379.0
mean      -7.9
std        7.9
min      -22.0
25%      -17.0
50%       -2.0
75%       -1.0
max       -1.0
dtype: float64


In [7]:
# Duration outliers
duration = pd.to_numeric(cleaned_opportunity_df[Fields.DURATION_MONTHS], errors="coerce")
print("Duration distribution (months):")
print(duration.describe().round(1))
print("\nOver 60 months:")
print(duration[duration > 60].describe().round(1))

Duration distribution (months):
count    7894.0
mean       10.2
std        17.3
min         0.0
25%         2.0
50%         4.0
75%        10.8
max       120.0
Name: project_duration_number_of_months, dtype: float64

Over 60 months:
count    152.0
mean      89.4
std       17.4
min       63.0
25%       78.0
50%       84.0
75%      101.5
max      120.0
Name: project_duration_number_of_months, dtype: float64


## 4. Categorical / scoring fields

Confirms `sales_stage` covers exactly the architecture's 7-stage table
(no unmapped values), `probability` stays in [0, 100], and the
`status` vs `status_reason` Won-detection disagreement is a 15-row corner case.

In [8]:
validate_categorical_fields(cleaned_opportunity_df)

,metric,value
0,n_total,8746
1,n_status_null,0
2,n_status_reason_null,51
3,n_sales_stage_null,0
4,n_probability_null,77
5,n_probability_negative,0
6,n_probability_above_100,0
7,n_distinct_status,3
8,n_distinct_status_reason,17
9,n_distinct_sales_stage,7


In [9]:
# status x status_reason crosstab — justifies the architecture's 'use status_reason' rule
pd.crosstab(
    cleaned_opportunity_df[Fields.STATUS],
    cleaned_opportunity_df[Fields.STATUS_REASON],
    margins=True,
    margins_name="total",
)

status_reason,Cancelled By CGI,Cancelled No Bid Decision,Cancelled No Bid Decision due to Client Feedback,Cancelled by Customer,Cancelled by Customer due to Covid19,Duplicated,LOST-Alliance Partner,LOST-Experience,LOST-Expertise,LOST-Price,LOST-Quality,LOST-Relationship,LOST-Solution,LOST-Speed/Avail. to Deliver,LOST-Unknown/Other,Open,Won,total
status,,,,,,,,,,,,,,,,,,
Closed,871,463,6,583,2,461,1,46,226,62,4,13,50,27,262,0,0,3077
Open,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,343,0,343
Won,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5275,5275
total,871,463,6,583,2,461,1,46,226,62,4,13,50,27,262,343,5275,8695


In [10]:
# sales_stage distribution (used for late_stage_deal_count and stage_weight mapping)
cleaned_opportunity_df[Fields.SALES_STAGE].value_counts(dropna=False)

sales_stage
6-Negotiation&Signature    5425
0-Lead/Suspect             1026
5-Client Decision           813
1-Identification            483
2-Qualification             388
4-Proposal                  312
3-Bid Planning              299
Name: count, dtype: int64

In [11]:
# probability distribution and null rate by status
probability = pd.to_numeric(cleaned_opportunity_df[Fields.PROBABILITY], errors="coerce")
print("Overall null rate:", probability.isna().mean().round(4))
print("\nNull rate by status:")
print(
    cleaned_opportunity_df.assign(_prob_null=probability.isna())
    .groupby(Fields.STATUS)["_prob_null"]
    .mean()
    .round(4)
)
print("\nValue distribution:")
print(probability.describe().round(2))

Overall null rate: 0.0088

Null rate by status:
status
Closed    0.0249
Open      0.0000
Won       0.0000
Name: _prob_null, dtype: float64

Value distribution:
count    8669.00
mean       73.28
std        36.49
min         0.00
25%        50.00
50%       100.00
75%       100.00
max       100.00
Name: probability, dtype: float64


## 4. Data-quality and merge flags

`cleaned_opportunity_df` carries four merge-provenance flags and eight
data-quality flags. `duplicate_flag` is 0 (the merge is clean on
`opportunity_id`); `unmatched_flag` is 1,067 legitimate opps2-only rows.
The two `flag_discrepancy_*` rows reconcile the flags against the field
validators.

In [12]:
# validate_quality_flags reads the cleaned frame's merge + data-quality flags.
# The two flag_discrepancy_* rows are cross-checks against the field
# validators: missing_revenue reconciles to 0; close_before_created shows the
# cleaner's flag (3,049) still uses the un-normalized comparison vs the
# corrected count (379).
validate_quality_flags(cleaned_opportunity_df)

F:\capstone\cgi-capstone\src\data_validator.py:713: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  within_5pct = (ratio.fillna(0) <= 0.05).sum()


,metric,value
0,n_total,8746
1,source_file_flag::opps2_base,8695
2,source_file_flag::opps1_exclusive,51
3,duplicate_flag_count,0
4,unmatched_flag_count,1067
5,opps1_exclusive_flag_count,51
6,missing_owner_flag_count,0
7,missing_probability_flag_count,77
8,invalid_probability_flag_count,0
9,missing_revenue_flag_count,0


## 5. status_reason outcome taxonomy

`status_reason` carries `Cancelled ...` and `Duplicated` values beyond
Won / Open / Lost. Sprint 2 decision is **Option B**: Cancelled is lost-like
(a real pipeline exit), Duplicated is carved into its own bucket and excluded
from win/loss counts. The profiling cell below is the evidence — the key fact
is that `status_reason == "Duplicated"` has **zero** overlap with the
merge-level `duplicate_flag` and is concentrated in just 3 owners, so it is a
CRM data-hygiene label, not a sales outcome. `classify_opportunity_outcome`
is the shared helper that implements this; see `docs/data_validation.md` §5.

In [13]:
# status_reason profiling — the evidence behind the Option B decision.
print("status value counts:")
print(cleaned_opportunity_df[Fields.STATUS].value_counts(dropna=False))
print("\nstatus_reason value counts:")
print(cleaned_opportunity_df[Fields.STATUS_REASON].value_counts(dropna=False))

# The key question: does status_reason == "Duplicated" coincide with the
# merge-level duplicate_flag? If it did, the dedup in build_owner_aggregates
# would already handle it. It does not.
sr = cleaned_opportunity_df[Fields.STATUS_REASON].astype("string").str.strip().str.lower()
is_duplicated = sr.eq("duplicated")
print("\nstatus_reason == 'Duplicated' vs duplicate_flag:")
print(pd.crosstab(is_duplicated, cleaned_opportunity_df["duplicate_flag"], dropna=False))
print(f"\nDuplicated rows: {int(is_duplicated.sum())}, "
      f"owners affected: {cleaned_opportunity_df.loc[is_duplicated, Fields.OWNER].nunique()}")

status value counts:
status
Won       5290
Closed    3091
Open       365
Name: count, dtype: int64

status_reason value counts:
status_reason
Won                                                 5275
Cancelled By CGI                                     871
Cancelled by Customer                                583
Cancelled No Bid Decision                            463
Duplicated                                           461
Open                                                 343
LOST-Unknown/Other                                   262
LOST-Expertise                                       226
LOST-Price                                            62
NaN                                                   51
LOST-Solution                                         50
LOST-Experience                                       46
LOST-Speed/Avail. to Deliver                          27
LOST-Relationship                                     13
Cancelled No Bid Decision due to Client Feedback       6
LOS

In [14]:
# classify_opportunity_outcome — the shared Option B taxonomy helper.
outcome = classify_opportunity_outcome(cleaned_opportunity_df)
print("Outcome bucket counts:")
print(outcome.value_counts(dropna=False))
print("\nReconciles to row count:", int(outcome.notna().sum()) == len(cleaned_opportunity_df))
print("\nThe 15 status==Won / null-status_reason rows now resolve to 'won':")
null_reason = cleaned_opportunity_df[Fields.STATUS_REASON].isna()
print(outcome[null_reason].value_counts())

Outcome bucket counts:
won          5290
lost         2630
duplicate     461
open          365
Name: count, dtype: int64

Reconciles to row count: True

The 15 status==Won / null-status_reason rows now resolve to 'won':
open    22
won     15
lost    14
Name: count, dtype: int64


## 6. Owner identity

24 distinct owners, no casing variants, zero owner-equals-manager rows.
Seven owners have fewer than 10 lifetime opportunities — their baselines
should carry a Low reliability flag.

In [15]:
validate_owner_identity(cleaned_opportunity_df)

,metric,value
0,n_total,8746.0
1,n_owner_null,0.0
2,n_owner_blank_string,0.0
3,n_manager_null,0.0
4,n_distinct_owner_raw,24.0
5,n_distinct_owner_normalized,24.0
6,n_owners_collapsing_under_normalization,0.0
7,n_rows_with_normalization_change,0.0
8,n_owner_equals_manager,0.0
9,pct_owner_equals_manager,0.0


In [16]:
# Owner deal counts (sorted, top + tail)
owner_counts = cleaned_opportunity_df[Fields.OWNER].value_counts()
print("Top 5 by lifetime opportunity count:")
print(owner_counts.head())
print("\nBottom 7 (Low reliability candidates):")
print(owner_counts.tail(7))

Top 5 by lifetime opportunity count:
opportunity_owner
Lee Northcott       4709
Cameron Foxworth    3641
Zara Fairfield        58
Logan Jameson         41
Hadley Randolph       32
Name: count, dtype: int64

Bottom 7 (Low reliability candidates):
opportunity_owner
Cameron Brennan    7
Morgan Kirkland    5
Val Iverson        2
Emery Iverson      2
Peyton Sterling    2
Quinn Calloway     1
Shawn Yamamoto     1
Name: count, dtype: int64


## 7. Owner aggregates

`build_owner_aggregates(cleaned_opportunity_df)` produces the ten columns
Freya's dashboard mock expects but Lyken's `director_df` does not yet emit —
including the Sprint 2 outcome counts (`open_`, `lost_`,
`duplicate_opportunity_count`) derived from `classify_opportunity_outcome`.
Duplicate join-key rows are deduplicated on `opportunity_id` before
aggregating. Lyken's score columns join onto this on `opportunity_owner`.

In [17]:
owner_aggregates = build_owner_aggregates(cleaned_opportunity_df)
print(f"{len(owner_aggregates)} owners")
print("baseline_reliability distribution:")
print(owner_aggregates["baseline_reliability"].value_counts())
owner_aggregates.sort_values("weighted_pipeline_revenue", ascending=False)

24 owners
baseline_reliability distribution:
baseline_reliability
Medium         13
Low             6
High            3
No baseline     2
Name: count, dtype: int64


,opportunity_owner,territory,late_stage_deal_count,weighted_pipeline_revenue,inferred_delivery_commitments,open_opportunity_count,lost_opportunity_count,duplicate_opportunity_count,quarters_of_data,baseline_reliability
14,Logan Jameson,<NA>,13,86953878.50,0,41,0,0,6,Medium
9,Hadley Randolph,CAN GTO,18,37753383.90,0,32,0,0,7,Medium
2,Blake Vickers,CAN GTO,0,33939948.40,0,30,0,0,5,Medium
1,Blake Radford,CAN ATL Ntl Svcs Credit Union,4,29051875.00,0,15,0,0,6,Medium
15,Morgan Kirkland,<NA>,0,27025000.00,0,5,0,0,2,Low
16,Oakley Dalton,CAN ATL Ntl Svcs Credit Union,2,20116848.00,0,11,0,0,5,Medium
8,Fern Westwood,APD GTO Canada,5,19955526.25,0,20,0,0,6,Medium
11,Jordan Jameson,CAN ATL Atlantic Metro,5,11371210.05,0,22,0,0,0,No baseline
13,Logan Easton,CAN ATL Ntl Svcs Credit Union,17,7648623.30,0,27,0,0,5,Medium
22,Zara Fairfield,CAN ATL Atlantic Metro,14,7099219.75,0,58,0,0,6,Medium


In [18]:
# Optional local export — overwrites previous run; data/processed/ is gitignored
from pathlib import Path
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
owner_aggregates.to_csv(out_dir / "owner_validation_summary.csv", index=False)
print("Wrote", out_dir / "owner_validation_summary.csv")

Wrote F:\capstone\cgi-capstone\data\processed\owner_validation_summary.csv


## 8. Field reliability for scoring

`build_field_reliability_report` rates each validated field as a scoring
input — `reliable` / `use_with_care` / `not_yet` — from computed null and
anomaly rates, plus a `usable_for_scoring` flag and the governing
`fallback_assumptions.yaml` rule. This replaces the Sprint 1 hand-rolled
colour-code cell. The human-readable version is `docs/data_validation.md` §8;
the targeted Lyken handoff is `docs/scoring_input_reliability_handoff.md`.

In [19]:
field_reliability = build_field_reliability_report(cleaned_opportunity_df)
print("reliability distribution:", dict(field_reliability["reliability"].value_counts()))
print("usable_for_scoring:", dict(field_reliability["usable_for_scoring"].value_counts()))
field_reliability

reliability distribution: {'reliable': np.int64(7), 'use_with_care': np.int64(5), 'not_yet': np.int64(1)}
usable_for_scoring: {True: np.int64(12), False: np.int64(1)}


,field,family,pct_null,n_anomalies,anomaly_kind,reliability,usable_for_scoring,fallback_rule,notes
0,total_estimated_revenue,revenue,0.58,291,zero_or_negative,use_with_care,True,revenue_hierarchy,Canonical CAD revenue; the small null gap is covered by ...
1,opportunity_estimated_revenue_base_cad,revenue,12.20,82,zero_or_negative,use_with_care,True,revenue_hierarchy,Now populated on matched rows as well as the opps1-exclu...
2,service_solution_estimated_revenue,revenue,43.83,148,zero_or_negative,not_yet,False,do_not_sum_service_solution_revenue,Service-line breakdown. Never sum onto authoritative_rev...
3,created_on,date,0.58,0,created_after_today,reliable,True,,Drives historical baselines and quarters_of_data; nearly...
4,close_date,date,4.17,379,close_before_created,use_with_care,True,delivery_window,Usable as the delivery_window fallback start. Carries a ...
5,revenue_start_date,date,12.49,0,none_null_is_the_concern,use_with_care,True,delivery_window,"Null overall but fully populated on Won deals, where the..."
6,project_duration_number_of_months,duration,9.74,195,zero_negative_or_over_60_months,use_with_care,True,missing_duration,"Nulls, non-positive values, and >60-month outliers all p..."
7,status,categorical,0.00,0,none,reliable,True,status_reason_outcome,"Fully populated, few distinct values. Outcome bucketing ..."
8,status_reason,categorical,0.58,15,won_flag_disagreement,reliable,True,status_reason_outcome,Canonical outcome field. classify_opportunity_outcome ha...
9,sales_stage,categorical,0.00,0,coverage_checked_in_categorical,reliable,True,late_stage_definition,Fully populated; stage coverage against the architecture...


---

**Use of Generative AI.** Anthropic Claude (Opus 4.7) was used for drafting and editing assistance on this notebook. All numerical findings come from running the validator functions on the team's `cleaned_opportunity_df`. No CGI data was sent to the tool.